## Partie de Paul

In [23]:
import requests
import pandas as pd
import time

# 1. Liste des années que tu veux récupérer
annees_a_recuperer = ["2019-2020", "2020-2021", "2021-2022", "2022-2023", "2023-2024", "2024-2025"]

all_results = []
dataset_id = "fr-en-calendrier-scolaire"
base_url = f"https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/{dataset_id}/records"

print("🚀 Récupération multi-années via 'refine'...")

for annee in annees_a_recuperer:
    offset = 0
    limit = 100
    print(f"📅 Traitement de l'année : {annee}")
    
    while True:
        params = {
            "limit": limit,
            "offset": offset,
            "refine": f"annee_scolaire:{annee}" # On cible l'année de la boucle
        }
        
        response = requests.get(base_url, params=params)
        if response.status_code != 200:
            print(f"❌ Erreur sur {annee} : {response.status_code}")
            break
            
        data = response.json()
        batch = data.get('results', [])
        
        if not batch:
            break
               
        all_results.extend(batch)
        
        # 'total_annee' nous dit combien il y a de lignes au total pour CETTE année-là
        total_annee = data.get('total_count', 0)
        
        # On calcule combien on en a déjà dans notre liste pour cette année précise
        recu_pour_annee = offset + len(batch)
        
        print(f"   📥 Page chargée : {recu_pour_annee} / {total_annee}")
        
        offset += limit
        time.sleep(0.2) # On peut aller un peu plus vite avec refine
    
# Envoi vers RDS (le reste de ton code ne change pas)
df_final = pd.json_normalize(all_results)
print(f"✅ Terminé ! {len(df_final)} lignes récupérées au total.")

🚀 Récupération multi-années via 'refine'...
📅 Traitement de l'année : 2019-2020
   📥 Page chargée : 100 / 229
   📥 Page chargée : 200 / 229
   📥 Page chargée : 229 / 229
📅 Traitement de l'année : 2020-2021
   📥 Page chargée : 100 / 228
   📥 Page chargée : 200 / 228
   📥 Page chargée : 228 / 228
📅 Traitement de l'année : 2021-2022
   📥 Page chargée : 100 / 232
   📥 Page chargée : 200 / 232
   📥 Page chargée : 232 / 232
📅 Traitement de l'année : 2022-2023
   📥 Page chargée : 100 / 229
   📥 Page chargée : 200 / 229
   📥 Page chargée : 229 / 229
📅 Traitement de l'année : 2023-2024
   📥 Page chargée : 100 / 232
   📥 Page chargée : 200 / 232
   📥 Page chargée : 232 / 232
📅 Traitement de l'année : 2024-2025
   📥 Page chargée : 100 / 235
   📥 Page chargée : 200 / 235
   📥 Page chargée : 235 / 235
✅ Terminé ! 1385 lignes récupérées au total.


In [24]:
df_final

,description,population,start_date,end_date,location,zones,annee_scolaire
0,Vacances de la Toussaint,-,2019-10-18T22:00:00+00:00,2019-11-03T23:00:00+00:00,Bordeaux,Zone A,2019-2020
1,Vacances de la Toussaint,-,2019-10-18T22:00:00+00:00,2019-11-03T23:00:00+00:00,Lyon,Zone A,2019-2020
2,Vacances de Noël,-,2019-12-20T23:00:00+00:00,2020-01-05T23:00:00+00:00,Limoges,Zone A,2019-2020
3,Vacances de Noël,-,2019-12-20T23:00:00+00:00,2020-01-05T23:00:00+00:00,Lyon,Zone A,2019-2020
4,Vacances d'Hiver,-,2020-02-21T23:00:00+00:00,2020-03-08T23:00:00+00:00,Grenoble,Zone A,2019-2020
...,...,...,...,...,...,...,...
1380,Vacances de la Toussaint,-,2024-10-21T22:00:00+00:00,2024-11-03T23:00:00+00:00,Guadeloupe,Guadeloupe,2024-2025
1381,Vacances d'Hiver austral,Enseignants,2025-07-04T22:00:00+00:00,2025-08-17T22:00:00+00:00,Réunion,Réunion,2024-2025
1382,Vacances de Pâques,-,2025-03-27T23:00:00+00:00,2025-04-13T22:00:00+00:00,Polynésie,Polynésie,2024-2025
1383,Grandes Vacances,Élèves du premier degré,2025-07-03T22:00:00+00:00,2025-08-12T22:00:00+00:00,Polynésie,Polynésie,2024-2025


In [25]:
import pandas as pd

# On part de ton df_final (issu de l'API)
# 1. S'assurer que les colonnes sont bien au format date
df_final['start_date'] = pd.to_datetime(df_final['start_date'])
df_final['end_date'] = pd.to_datetime(df_final['end_date'])

# 2. Créer une fonction qui génère la liste des dates entre start et end
def generate_dates(row):
    return pd.date_range(start=row['start_date'], end=row['end_date']).tolist()

# 3. Appliquer la fonction pour créer une nouvelle colonne "liste_dates"
df_final['date'] = df_final.apply(generate_dates, axis=1)

# 4. L'étape magique : EXPLODE
# Cette fonction transforme chaque élément de la liste en une nouvelle ligne
df_unfolded = df_final.explode('date')

# 5. On peut maintenant supprimer les anciennes colonnes start/end si on veut
df_unfolded = df_unfolded.drop(columns=['start_date', 'end_date'])

print(f"✅ Le calendrier est déplié. Nouveau nombre de lignes : {len(df_unfolded)}")

✅ Le calendrier est déplié. Nouveau nombre de lignes : 36629


In [26]:
df_unfolded

,description,population,location,zones,annee_scolaire,date
0,Vacances de la Toussaint,-,Bordeaux,Zone A,2019-2020,2019-10-18 22:00:00+00:00
0,Vacances de la Toussaint,-,Bordeaux,Zone A,2019-2020,2019-10-19 22:00:00+00:00
0,Vacances de la Toussaint,-,Bordeaux,Zone A,2019-2020,2019-10-20 22:00:00+00:00
0,Vacances de la Toussaint,-,Bordeaux,Zone A,2019-2020,2019-10-21 22:00:00+00:00
0,Vacances de la Toussaint,-,Bordeaux,Zone A,2019-2020,2019-10-22 22:00:00+00:00
...,...,...,...,...,...,...
1384,Grandes Vacances,Enseignants du premier degré,Polynésie,Polynésie,2024-2025,2025-08-06 22:00:00+00:00
1384,Grandes Vacances,Enseignants du premier degré,Polynésie,Polynésie,2024-2025,2025-08-07 22:00:00+00:00
1384,Grandes Vacances,Enseignants du premier degré,Polynésie,Polynésie,2024-2025,2025-08-08 22:00:00+00:00
1384,Grandes Vacances,Enseignants du premier degré,Polynésie,Polynésie,2024-2025,2025-08-09 22:00:00+00:00
